In [0]:
----------------------------------------------
CREATE OR REPLACE TABLE `capgemini_academy`.`silver`.`pedidos_autoloader` AS
    SELECT
        TRY_CAST(pedido_id AS BIGINT) AS pedido_id,
        TRIM(cliente_id) AS cliente_id,
        CAST(quantidade AS INT) AS quantidade,
        CAST(valor_total AS DECIMAL(10,2)) AS valor_total,
        TO_DATE(data_pedido) AS data_pedido
    FROM `capgemini_academy`.`bronze`.`pedidos_autoloader`
    WHERE 1=1
      AND pedido_id IS NOT NULL
      AND quantidade > 0;
----------------------------------------------
CREATE OR REPLACE TABLE `capgemini_academy`.`silver`.`pedidos_autoloader_v2` AS
    WITH ranked AS (
        SELECT *,
                ROW_NUMBER() OVER (PARTITION BY pedido_id ORDER BY _ingested_at DESC) AS rn
        FROM `capgemini_academy`.`bronze`.`pedidos_autoloader`)

    SELECT
        TRY_CAST(pedido_id AS BIGINT) AS pedido_id,
        TRIM(cliente_id) AS cliente_id,
        TRIM(produto_id) AS produto_id,
        TRY_CAST(quantidade AS INT) AS quantidade,
        TRY_CAST(valor_total AS DECIMAL(10,2)) AS valor_total,
        TO_DATE(data_pedido) AS data_pedido,
        CURRENT_TIMESTAMP() AS silver_ingest_ts
    FROM ranked
    WHERE 1=1
      AND rn = 1
      AND pedido_id IS NOT NULL
      AND quantidade > 0;
----------------------------------------------
SELECT * FROM `capgemini_academy`.`silver`.`pedidos_autoloader`;
SELECT * FROM `capgemini_academy`.`silver`.`pedidos_autoloader_v2`;
